# 00 — Data preview and integrity checks
---
This notebook performs initial structural and quality-control checks
for the integrated DMD master table.

Goals:
- verify that the master table loads correctly;
- inspect schema, types, and missingness;
- check genomic coordinate integrity;
- inspect clinical labels and DMD-specificity;
- identify potential red flags before downstream annotation and EDA.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px

PROCESSED = Path("../data/processed")
df = pd.read_csv(PROCESSED / "DMD_variants_master.csv")

print("Loaded:", PROCESSED / "DMD_variants_master.csv")
print("Shape:", df.shape)

Loaded: ../data/processed/DMD_variants_master.csv
Shape: (11308, 25)


## 1. Basic structure

We first inspect the size, schema and first rows of the integrated master table.

In [2]:
df.shape

(11308, 25)

In [3]:
df.columns.tolist()

['var_id',
 'rsid',
 'chr',
 'pos',
 'interval_length',
 'protein_change',
 'var_type',
 'clinvar_consequence',
 'clinvar_class',
 'clinvar_class_simple',
 'is_pathogenic',
 'phenotype_group',
 'condition_raw',
 'ref',
 'alt',
 'allele_freq',
 'in_gnomad',
 'af_high',
 'hemizygote_count',
 'homozygote_count',
 'ensembl_consequence',
 'aa_change',
 'aa_pos',
 'revel',
 'meta_lr']

In [4]:
df.dtypes

var_id                    int64
rsid                     object
chr                      object
pos                     float64
interval_length         float64
protein_change           object
var_type                 object
clinvar_consequence      object
clinvar_class            object
clinvar_class_simple     object
is_pathogenic              bool
phenotype_group          object
condition_raw            object
ref                      object
alt                      object
allele_freq             float64
in_gnomad                object
af_high                  object
hemizygote_count        float64
homozygote_count        float64
ensembl_consequence      object
aa_change                object
aa_pos                  float64
revel                   float64
meta_lr                 float64
dtype: object

In [5]:
df.head()

,var_id,rsid,chr,pos,interval_length,protein_change,var_type,clinvar_consequence,clinvar_class,clinvar_class_simple,...,allele_freq,in_gnomad,af_high,hemizygote_count,homozygote_count,ensembl_consequence,aa_change,aa_pos,revel,meta_lr
0,488014,NaN,X,1.0,156040894.0,NaN,Duplication,NaN,Pathogenic,pathogenic,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,146764,NaN,X,10001.0,156020894.0,NaN,copy number loss,NaN,Pathogenic,pathogenic,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,160983,NaN,X,10679.0,156002488.0,NaN,copy number loss,NaN,Pathogenic,pathogenic,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,160897,NaN,X,10679.0,156011527.0,NaN,copy number gain,NaN,Pathogenic,pathogenic,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,160890,NaN,X,10679.0,156011527.0,NaN,copy number loss,NaN,Pathogenic,pathogenic,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Missingness overview

We inspect missing values globally and in key column groups.

In [6]:
missing_global = (df.isna().mean() * 100).sort_values(ascending=False).round(2)
missing_global

meta_lr                 79.90
revel                   79.90
interval_length         77.74
aa_change               68.78
in_gnomad               68.74
homozygote_count        68.74
hemizygote_count        68.74
af_high                 68.74
ref                     68.74
alt                     68.74
allele_freq             68.74
aa_pos                  68.67
ensembl_consequence     62.83
protein_change          56.94
rsid                    29.44
clinvar_consequence     14.90
condition_raw            2.25
clinvar_class            2.24
phenotype_group          0.00
is_pathogenic            0.00
clinvar_class_simple     0.00
var_type                 0.00
pos                      0.00
chr                      0.00
var_id                   0.00
dtype: float64

In [7]:
key_cols = ["chr", "pos", "clinvar_class_simple", "is_pathogenic", "var_type"]
(df[key_cols].notna().mean() * 100).round(2)

chr                     100.0
pos                     100.0
clinvar_class_simple    100.0
is_pathogenic           100.0
var_type                100.0
dtype: float64

In [8]:
ensembl_cols = [
    "ensembl_consequence",
    "aa_change",
    "aa_pos",
    "revel",
    "meta_lr",
    "mutation_assessor"
]

existing_ensembl_cols = [c for c in ensembl_cols if c in df.columns]
(df[existing_ensembl_cols].notna().mean() * 100).round(2)

ensembl_consequence    37.17
aa_change              31.22
aa_pos                 31.33
revel                  20.10
meta_lr                20.10
dtype: float64

In [9]:
gnomad_cols = [
    "allele_freq",
    "in_gnomad",
    "af_high",
    "hemizygote_count",
    "homozygote_count"
]

existing_gnomad_cols = [c for c in gnomad_cols if c in df.columns]
(df[existing_gnomad_cols].notna().mean() * 100).round(2)

allele_freq         31.26
in_gnomad           31.26
af_high             31.26
hemizygote_count    31.26
homozygote_count    31.26
dtype: float64

## 3. Coordinate integrity

Since this project focuses on DMD, variants should largely map to the DMD genomic region on chromosome X.

In [10]:
df["chr"].value_counts(dropna=False)

chr
X    11308
Name: count, dtype: int64

In [11]:
pd.to_numeric(df["pos"], errors="coerce").describe()

count    1.130800e+04
mean     3.173998e+07
std      3.476588e+06
min      1.000000e+00
25%      3.162781e+07
50%      3.236295e+07
75%      3.250302e+07
max      3.334066e+07
Name: pos, dtype: float64

In [12]:
pos_num = pd.to_numeric(df["pos"], errors="coerce")

suspicious_pos_mask = (pos_num < 30_000_000) | (pos_num > 34_000_000)
suspicious_pos_fraction = suspicious_pos_mask.mean() * 100

suspicious_pos_fraction

np.float64(1.4326140785284756)

In [13]:
df.loc[suspicious_pos_mask, ["var_id", "chr", "pos", "interval_length", "var_type", "condition_raw"]].head(20)

,var_id,chr,pos,interval_length,var_type,condition_raw
0,488014,X,1.0,156040894.0,Duplication,Schizophrenia|Autism
1,146764,X,10001.0,156020894.0,copy number loss,See cases
2,160983,X,10679.0,156002488.0,copy number loss,See cases
3,160897,X,10679.0,156011527.0,copy number gain,See cases
4,160890,X,10679.0,156011527.0,copy number loss,See cases
5,148082,X,10679.0,49146835.0,copy number loss,See cases
6,148051,X,10679.0,76409826.0,copy number gain,See cases
7,148018,X,10679.0,52203052.0,copy number loss,See cases
8,146239,X,10679.0,36175956.0,copy number loss,See cases
9,145623,X,10679.0,52847126.0,copy number gain,See cases


In [14]:
pd.to_numeric(df["interval_length"], errors="coerce").describe()

count    2.517000e+03
mean     6.130720e+06
std      2.707346e+07
min      1.000000e+00
25%      3.000000e+00
50%      5.520100e+04
75%      2.763100e+05
max      1.560409e+08
Name: interval_length, dtype: float64

In [15]:
interval_num = pd.to_numeric(df["interval_length"], errors="coerce")
large_interval_mask = interval_num > 1_000_000

df.loc[large_interval_mask, ["var_id", "chr", "pos", "interval_length", "var_type", "condition_raw"]].head(20)

,var_id,chr,pos,interval_length,var_type,condition_raw
0,488014,X,1.0,156040894.0,Duplication,Schizophrenia|Autism
1,146764,X,10001.0,156020894.0,copy number loss,See cases
2,160983,X,10679.0,156002488.0,copy number loss,See cases
3,160897,X,10679.0,156011527.0,copy number gain,See cases
4,160890,X,10679.0,156011527.0,copy number loss,See cases
5,148082,X,10679.0,49146835.0,copy number loss,See cases
6,148051,X,10679.0,76409826.0,copy number gain,See cases
7,148018,X,10679.0,52203052.0,copy number loss,See cases
8,146239,X,10679.0,36175956.0,copy number loss,See cases
9,145623,X,10679.0,52847126.0,copy number gain,See cases


## 4. Clinical label integrity

We now inspect clinical classifications and phenotype-related columns.

In [16]:
df["clinvar_class_simple"].value_counts(dropna=False)

clinvar_class_simple
vus                  3251
likely_benign        3062
pathogenic           2858
other                 937
benign                640
likely_pathogenic     560
Name: count, dtype: int64

In [17]:
df["is_pathogenic"].value_counts(dropna=False)

is_pathogenic
False    7890
True     3418
Name: count, dtype: int64

In [18]:
df["phenotype_group"].value_counts(dropna=False)

phenotype_group
DMD      7807
other    2478
BMD      1023
Name: count, dtype: int64

In [19]:
df["condition_raw"].dropna().sample(min(20, df["condition_raw"].dropna().shape[0]), random_state=42).tolist()

['not specified',
 'Duchenne muscular dystrophy',
 'Duchenne muscular dystrophy',
 'Duchenne muscular dystrophy',
 'Dilated cardiomyopathy 3B',
 'Duchenne muscular dystrophy',
 'Duchenne muscular dystrophy',
 'not provided',
 'Duchenne muscular dystrophy',
 'not provided',
 'Duchenne muscular dystrophy',
 'Duchenne muscular dystrophy',
 'Duchenne muscular dystrophy',
 'Duchenne muscular dystrophy',
 'Duchenne muscular dystrophy',
 'not provided',
 'Duchenne muscular dystrophy',
 'Elevated circulating creatine kinase concentration',
 'Cardiovascular phenotype|not provided|Duchenne muscular dystrophy|Left ventricular noncompaction cardiomyopathy',
 'not provided|Duchenne muscular dystrophy']

In [20]:
mask_not_dmd = ~df["condition_raw"].fillna("").str.contains(
    "duchenne|becker|dmd|bmd",
    case=False,
    regex=True
)

mask_not_dmd.mean() * 100

np.float64(21.71913689423417)

In [21]:
df["condition_raw"].fillna("NA").value_counts().head(20)

condition_raw
Duchenne muscular dystrophy                                                                      5841
not provided                                                                                     1216
Becker muscular dystrophy, Cardiomyopathy, Duchenne muscular dystrophy, Dystrophin deficiency     559
not provided|Duchenne muscular dystrophy                                                          340
Duchenne muscular dystrophy|not provided                                                          291
Duchenne muscular dystrophy|Cardiovascular phenotype                                              281
Cardiovascular phenotype                                                                          274
NA                                                                                                254
Cardiovascular phenotype|Duchenne muscular dystrophy                                              239
See cases                                                           

## 5. Variant-type overview

We inspect the distribution of variant types and consequences.

In [22]:
df["var_type"].value_counts(dropna=False).head(20)

var_type
single nucleotide variant    8389
Deletion                     1499
Duplication                   772
copy number loss              257
Microsatellite                155
copy number gain              122
Insertion                      56
Indel                          49
Inversion                       6
Haplotype                       2
Complex                         1
Name: count, dtype: int64

In [23]:
df["clinvar_consequence"].value_counts(dropna=False).head(20)

clinvar_consequence
missense variant                          2888
intron variant                            2235
NaN                                       1685
synonymous variant                        1627
nonsense                                   724
frameshift variant                         596
missense variant|5 prime UTR variant       335
splice donor variant                       218
synonymous variant|5 prime UTR variant     209
splice acceptor variant                    196
frameshift variant|5 prime UTR variant      95
missense variant|intron variant             86
nonsense|5 prime UTR variant                72
3 prime UTR variant                         55
synonymous variant|intron variant           46
frameshift variant|intron variant           36
inframe_deletion                            28
5 prime UTR variant|intron variant          26
nonsense|intron variant                     22
3 prime UTR variant|intron variant          16
Name: count, dtype: int64

## 6. Red flags

We combine several rules to identify suspicious or potentially off-target records:
- positions outside the canonical DMD region on chrX;
- very large interval lengths;
- non-DMD-like condition labels.

In [24]:
red_flags = pd.Series(False, index=df.index)

if "pos" in df.columns:
    pos_num = pd.to_numeric(df["pos"], errors="coerce")
    red_flags |= (pos_num < 30_000_000) | (pos_num > 34_000_000)

if "interval_length" in df.columns:
    interval_num = pd.to_numeric(df["interval_length"], errors="coerce")
    red_flags |= interval_num > 1_000_000

if "condition_raw" in df.columns:
    red_flags |= df["condition_raw"].fillna("").str.contains(
        "schizophrenia|autism",
        case=False,
        regex=True
    )

red_flag_df = df.loc[
    red_flags,
    [c for c in ["var_id", "chr", "pos", "interval_length", "var_type", "condition_raw"] if c in df.columns]
]

print("Red-flag rows:", red_flag_df.shape[0])
red_flag_df.head(30)

Red-flag rows: 209


,var_id,chr,pos,interval_length,var_type,condition_raw
0,488014,X,1.0,156040894.0,Duplication,Schizophrenia|Autism
1,146764,X,10001.0,156020894.0,copy number loss,See cases
2,160983,X,10679.0,156002488.0,copy number loss,See cases
3,160897,X,10679.0,156011527.0,copy number gain,See cases
4,160890,X,10679.0,156011527.0,copy number loss,See cases
5,148082,X,10679.0,49146835.0,copy number loss,See cases
6,148051,X,10679.0,76409826.0,copy number gain,See cases
7,148018,X,10679.0,52203052.0,copy number loss,See cases
8,146239,X,10679.0,36175956.0,copy number loss,See cases
9,145623,X,10679.0,52847126.0,copy number gain,See cases


## 7. Quick sanity plots

A few simple visual checks are added here for quick inspection.

In [25]:
miss_plot = missing_global.reset_index()
miss_plot.columns = ["column", "missing_percent"]

fig = px.bar(
    miss_plot.head(15),
    x="column",
    y="missing_percent",
    title="Top 15 columns by missingness",
)
fig.update_xaxes(tickangle=-45)
fig.show()

In [26]:
chr_plot = df["chr"].fillna("NA").value_counts().reset_index()
chr_plot.columns = ["chr", "count"]

fig = px.bar(
    chr_plot,
    x="chr",
    y="count",
    title="Chromosome distribution in master table"
)
fig.show()

In [27]:
if "pos" in df.columns:
    fig = px.histogram(
        df.dropna(subset=["pos"]),
        x="pos",
        nbins=100,
        title="Distribution of genomic positions"
    )
    fig.show()

In [28]:
if "interval_length" in df.columns:
    interval_clean = pd.to_numeric(df["interval_length"], errors="coerce").dropna()

    if len(interval_clean) > 0:
        fig = px.histogram(
            interval_clean,
            nbins=80,
            title="Distribution of interval lengths"
        )
        fig.show()

## 8. Summary

The integrated DMD master dataset contains 11,308 variants across 26 columns.

All variants map to chromosome X, confirming correct gene-level filtering.

Genomic coordinate completeness is high, with the majority of variants having valid chr/pos values.
Approximately 1.43% of variants show suspicious genomic positions outside the canonical DMD locus (~30–34 Mb on chrX).

Further inspection reveals that these cases are strongly associated with very large interval lengths, suggesting that they correspond to structural variants (e.g. large deletions or imprecise CNV calls) rather than simple point mutations.

Phenotype distribution is dominated by Duchenne muscular dystrophy (DMD, around 69%), followed by Becker muscular dystrophy (BMD, around 9%) and other or unspecified phenotypes (around 22%).
This is consistent with the clinical representation of dystrophinopathies in public databases.

A small number of records contain non-DMD-related condition labels (e.g. neurological disorders), likely reflecting database-level annotation noise.

Overall, the dataset is of high quality and suitable for downstream annotation and exploratory analysis, with minor expected artifacts related to large structural variants.

In [29]:
summary = {
    "n_rows": df.shape[0],
    "n_columns": df.shape[1],
    "chr_filled_percent": round(df["chr"].notna().mean() * 100, 2) if "chr" in df.columns else None,
    "pos_filled_percent": round(pd.to_numeric(df["pos"], errors="coerce").notna().mean() * 100, 2) if "pos" in df.columns else None,
    "suspicious_pos_percent": round(suspicious_pos_fraction, 2) if "pos" in df.columns else None,
    "red_flag_rows": int(red_flag_df.shape[0]),
}

pd.Series(summary)

n_rows                    11308.00
n_columns                    25.00
chr_filled_percent          100.00
pos_filled_percent          100.00
suspicious_pos_percent        1.43
red_flag_rows               209.00
dtype: float64